# 003 · TOML with tomllib

> **Standard Library › Data Formats** · Python 3.14

TOML is the configuration format of the modern Python ecosystem — `pyproject.toml` describes every project, including this one. The `tomllib` module (added in Python 3.11) **parses** TOML into plain Python dicts; it deliberately cannot write it.

**In this notebook you will learn:**

- how TOML compares to `appsettings.json`
- `tomllib.loads` (strings) and `tomllib.load` (files — opened in **binary** mode)
- how TOML types map to Python, including native dates and times
- how to read the typical sections of a `pyproject.toml`
- why there is no writer, and how errors are reported (`TOMLDecodeError`)

## .NET mental model

| .NET | Python |
|---|---|
| `appsettings.json` | `pyproject.toml` / your own `config.toml` |
| `ConfigurationBuilder().AddJsonFile(...)` | `tomllib.load(f)` → a nested `dict` |
| `config["Logging:LogLevel:Default"]` | `config["logging"]["level"]` |
| `config.GetSection("Db").Get<DbOptions>()` | build a dataclass from the sub-dict yourself |
| Tomlyn (NuGet) — TOML for .NET | `tomllib` (read) + `tomli-w` (third-party, write) |

There is no TOML parser in the .NET BCL, and no equivalent of the `IConfiguration` layering (environment overrides, reload-on-change) in `tomllib` — it is just a parser, like `json.loads`.

## A quick look at TOML

TOML is like an INI file with real types: strings, integers, floats, booleans, dates, arrays and nested tables.

```toml
# comments are allowed (unlike JSON)
name = "inventory-service"
debug = false
max_connections = 50

[database]                  # a table = a nested object
host = "localhost"
port = 5432

[[workers]]                 # an array of tables
name = "importer"
[[workers]]
name = "mailer"
```

The equivalent `appsettings.json` would need quotes everywhere and could not contain comments (officially).

## Parsing a string: `tomllib.loads`

In [ ]:
import tomllib

config_text = '''
name = "inventory-service"
debug = false
max_connections = 50
ratio = 0.75

[database]
host = "localhost"
port = 5432

[[workers]]
name = "importer"
threads = 4

[[workers]]
name = "mailer"
threads = 1
'''

config = tomllib.loads(config_text)
print(type(config).__name__)
print(config["name"], config["debug"], config["max_connections"])
print(config["database"])
print([w["name"] for w in config["workers"]])

The result is an ordinary `dict`, so everything from *(see Syntax › Collections › 003 · Dictionaries)* applies. `.get()` with a default is the idiomatic way to treat a setting as optional:

In [ ]:
timeout = config["database"].get("timeout_seconds", 30)
print(timeout)

## Reading a file: `tomllib.load` needs binary mode

TOML files are always UTF-8, so `tomllib` decodes them itself and insists on a file opened with `"rb"`.

In [ ]:
import tempfile
from pathlib import Path

with tempfile.TemporaryDirectory() as tmp:
    path = Path(tmp) / "config.toml"
    path.write_text('[server]\nport = 8080\n', encoding="utf-8")

    with path.open("rb") as f:          # ✅ binary mode
        print(tomllib.load(f))

    try:
        with path.open(encoding="utf-8") as f:   # ⚠️ text mode
            tomllib.load(f)
    except TypeError as e:
        print(f"{type(e).__name__}: {e}")

**Tip:** `tomllib.loads(path.read_text(encoding="utf-8"))` also works if you already have a text string.

## Type mapping

| TOML | Python |
|---|---|
| table / inline table | `dict` |
| array / array of tables | `list` |
| string | `str` |
| integer (incl. `0xFF`, `1_000`) | `int` |
| float (incl. `inf`, `nan`) | `float` |
| boolean | `bool` |
| offset date-time `2025-03-01T08:00:00Z` | `datetime` with `tzinfo` |
| local date-time `2025-03-01T08:00:00` | naive `datetime` |
| local date `2025-03-01` | `date` |
| local time `08:00:00` | `time` |

Unlike JSON, dates and times are first-class — no `default=` hooks needed.

In [ ]:
types_text = '''
released   = 2025-03-01
starts_at  = 08:30:00
local_dt   = 2025-03-01T08:30:00
utc_dt     = 2025-03-01T08:30:00Z
offset_dt  = 2025-03-01T08:30:00+02:00
mask       = 0xFF
population = 1_000_000
limits     = { min = 1, max = 10 }
'''
for key, value in tomllib.loads(types_text).items():
    print(f"{key:10} {type(value).__name__:9} {value!r}")

Like `json.loads`, `tomllib.loads` accepts `parse_float=` — use `Decimal` when floats represent money:

In [ ]:
from decimal import Decimal

print(tomllib.loads("price = 19.99", parse_float=Decimal))

## Reading a `pyproject.toml`

A `pyproject.toml` has a standard `[project]` table (PEP 621) plus a `[tool.*]` namespace where each tool keeps its settings. Here is a representative sample embedded as a string:

In [ ]:
pyproject_text = '''
[project]
name = "inventory-service"
version = "1.4.0"
requires-python = ">=3.12"
dependencies = [
    "httpx>=0.27",
    "pydantic>=2.7",
]

[project.optional-dependencies]
dev = ["pytest>=8"]

[project.scripts]
inventory = "inventory.cli:main"

[tool.ruff]
line-length = 100

[tool.pytest.ini_options]
testpaths = ["tests"]
'''

pyproject = tomllib.loads(pyproject_text)
project = pyproject["project"]
print(project["name"], project["version"], project["requires-python"])
print(project["dependencies"])
print(pyproject["tool"]["ruff"]["line-length"])
print(pyproject["tool"]["pytest"]["ini_options"]["testpaths"])

Dotted table headers like `[tool.pytest.ini_options]` become nested dicts. Keys may contain `-`, so always use `["requires-python"]`, never attribute access.

**Tip:** to read a project's *installed* metadata (version, dependencies) at runtime, use `importlib.metadata.version("package")` instead of parsing `pyproject.toml` — the TOML file usually isn't shipped with an installed package.

## Mapping a table to a typed object

As with `IOptions<T>` binding in .NET, a dataclass gives you named, typed access. Python won't bind it for you, but `**` unpacking makes it a one-liner when the keys match the field names:

```csharp
// C#
var db = config.GetSection("Database").Get<DatabaseOptions>();
```

In [ ]:
from dataclasses import dataclass

@dataclass(frozen=True)
class DatabaseOptions:
    host: str
    port: int = 5432
    timeout_seconds: int = 30

db = DatabaseOptions(**config["database"])
print(db)

## There is no writer

`tomllib` is read-only by design: configuration is usually written by humans, and a round-trip writer that preserves comments and formatting is a much bigger job. If you need to *write* TOML, use a third-party package:

- `tomli-w` — a simple `dumps`/`dump` writer for plain data
- `tomlkit` — style-preserving round-trip editing (keeps comments and layout)

In [ ]:
print(hasattr(tomllib, "dumps"), [name for name in dir(tomllib) if not name.startswith("_")])

## Errors: `TOMLDecodeError`

Invalid TOML raises `tomllib.TOMLDecodeError`, a subclass of `ValueError`, with the bare message in `.msg` and the position in `.lineno` / `.colno`. TOML is stricter than you might expect: duplicate keys are an error (in JSON the last one silently wins).

In [ ]:
bad_documents = [
    'port = 8080\nport = 9090',        # duplicate key
    'name = unquoted',                 # strings must be quoted
    '[server]\nhost = "a"\n[server]',  # table defined twice
]
for doc in bad_documents:
    try:
        tomllib.loads(doc)
    except tomllib.TOMLDecodeError as e:
        print(f"{type(e).__name__}: {e.msg} -> line {e.lineno}, col {e.colno}")

## Summary

| Task | C# / .NET | Python |
|---|---|---|
| Parse a config string | Tomlyn `Toml.ToModel(text)` | `tomllib.loads(text)` |
| Parse a config file | `AddJsonFile("appsettings.json")` | `with open(p, "rb") as f: tomllib.load(f)` |
| Read a nested value | `config["Db:Host"]` | `cfg["db"]["host"]` |
| Optional value with default | `config.GetValue("X", 30)` | `cfg.get("x", 30)` |
| Bind to a typed object | `GetSection("Db").Get<DbOptions>()` | `DbOptions(**cfg["db"])` |
| Exact decimals | `decimal` binding | `parse_float=Decimal` |
| Write TOML | Tomlyn `Toml.FromModel(model)` | third-party `tomli-w` / `tomlkit` |
| Parse error | `FormatException` / `TomlException` | `tomllib.TOMLDecodeError` |

## Exercises

1. Parse the TOML below and print each server's name and full address as `name -> host:port`, using a default port of `80` when it is missing.
   ```toml
   [[servers]]
   name = "alpha"
   host = "10.0.0.1"
   port = 8080
   [[servers]]
   name = "beta"
   host = "10.0.0.2"
   ```
2. Write a function `load_config(path)` that reads a TOML file, and returns `{}` if the file does not exist. Test it inside a `TemporaryDirectory` with one existing and one missing file.
3. Parse `maintenance = 2025-06-01T22:00:00Z` and compute how many hours after `2025-06-01T12:00:00+00:00` the maintenance starts.

In [ ]:
# Your solutions here

<details>
<summary>Show solutions</summary>

```python
import tempfile
import tomllib
from datetime import datetime
from pathlib import Path

# 1 — arrays of tables become a list of dicts
servers_toml = '''
[[servers]]
name = "alpha"
host = "10.0.0.1"
port = 8080
[[servers]]
name = "beta"
host = "10.0.0.2"
'''
for s in tomllib.loads(servers_toml)["servers"]:
    print(f"{s['name']} -> {s['host']}:{s.get('port', 80)}")

# 2 — binary mode, and EAFP for the missing file
def load_config(path: Path) -> dict:
    try:
        with path.open("rb") as f:
            return tomllib.load(f)
    except FileNotFoundError:
        return {}

with tempfile.TemporaryDirectory() as tmp:
    existing = Path(tmp) / "app.toml"
    existing.write_text('debug = true\n', encoding="utf-8")
    print(load_config(existing))                  # {'debug': True}
    print(load_config(Path(tmp) / "missing.toml"))  # {}

# 3 — offset date-times come back as aware datetimes
starts = tomllib.loads("maintenance = 2025-06-01T22:00:00Z")["maintenance"]
noon = datetime.fromisoformat("2025-06-01T12:00:00+00:00")
print((starts - noon).total_seconds() / 3600)   # 10.0
```

</details>